<a href="https://colab.research.google.com/github/Vivian1312/StyleMatchAI/blob/main/StyleMatch_ejecuci%C3%B3n.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Integrar Fashion-CLIP en el entorno de Django

In [ ]:
import os
import numpy as np
from PIL import Image
import torch
from transformers import CLIPProcessor, CLIPModel
# Asegúrate de configurar el entorno de Django si ejecutas esto como script independiente
# os.environ.setdefault('DJANGO_SETTINGS_MODULE', 'stylematch.settings')
# import django
# django.setup()

def extraer_vectores_fashion_clip(directorio_imagenes):
    print("Cargando el modelo Fashion-CLIP...")
    # Usando el modelo base de Fashion-CLIP disponible en HuggingFace (ej. 'patrickjohncyh/fashion-clip')
    model_id = "patrickjohncyh/fashion-clip"
    model = CLIPModel.from_pretrained(model_id)
    processor = CLIPProcessor.from_pretrained(model_id)

    model.eval()
    vectores_registrados = {}

    print(f"Procesando imágenes desde: {directorio_imagenes}")

    for filename in os.listdir(directorio_imagenes):
        if filename.lower().endswith(('.png', '.jpg', '.jpeg')):
            image_path = os.path.join(directorio_imagenes, filename)
            try:
                image = Image.open(image_path).convert("RGB")

                # Procesar imagen para el modelo
                inputs = processor(images=image, return_tensors="pt")

                with torch.no_grad():
                    image_features = model.get_image_features(**inputs)

                # Normalizar el vector (buena práctica para similitud de coseno)
                image_features = image_features / image_features.norm(p=2, dim=-1, keepdim=True)
                vector_np = image_features.cpu().numpy().flatten()

                vectores_registrados[filename] = vector_np
                print(f"[ÉXITO] Vector extraído para: {filename}")

            except Exception as e:
                print(f"[ERROR] No se pudo procesar {filename}: {e}")

    return vectores_registrados

if __name__ == "__main__":
    # Ajusta esta ruta a tu carpeta de imágenes autorizadas (ej. media/prendas)
    path_imagenes = "media/prendas"
    if os.path.exists(path_imagenes):
        vectores = extraer_vectores_fashion_clip(path_imagenes)
        # Guardar resultados o actualizar la base de datos SQLite aquí
        np.save("vectores_prendas.npy", vectores)
        print("Extracción completada y guardada con trazabilidad.")
    else:
        print(f"La ruta {path_imagenes} no existe. Asegúrate de tener imágenes cargadas.")